# EDA regional — Análisis A (producción por piso ecológico)

Exploración **descriptiva** de `dataset_regional.csv` (agregado región × piso × mes, cultivos Pareto-80 sumados).

**Objetivos (alineados al paper):**
1. Caracterizar volumen y estacionalidad productiva por unidad territorial.
2. Describir perfiles climáticos por piso ecológico.
3. Explorar co-movimientos clima–producción (Pearson, sin inferencia causal).
4. Documentar eventos extremos: sequía Puno 2022 y El Niño costero 2023–2024.

**Limitaciones explícitas:**
- Correlación ≠ causalidad; sin desestacionalización ni corrección Benjamini–Hochberg.
- Cultivos del mismo (región, piso, distrito) comparten clima idéntico.
- Producción en toneladas (volumen), no rendimiento t/ha.

**Unidades:** `radiacion_solar` MJ/m²/día; `precipitacion` mm/día; `humedad_suelo` índice 0–1.

**Downstream:** hallazgos regionales contextualizan `05_eda_por_cultivo.ipynb` y `06_clustering_cultivos.ipynb`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif ROOT.name == "SCRIPTS":
    ROOT = ROOT.parent
elif not (ROOT / "OUTPUTS").exists() and (ROOT.parent / "OUTPUTS").exists():
    ROOT = ROOT.parent

RUTA_OUTPUT = ROOT / "OUTPUTS"
RUTA_OUTPUT.mkdir(parents=True, exist_ok=True)
RUTA_FIGURES = RUTA_OUTPUT / "figures"
RUTA_FIGURES.mkdir(parents=True, exist_ok=True)
DATASET_REGIONAL = RUTA_OUTPUT / "dataset_regional.csv"

if not DATASET_REGIONAL.exists():
    raise FileNotFoundError(
        f"No se encontró {DATASET_REGIONAL}. Ejecutar primero 03_build_dataset_integrado.ipynb"
    )

df = pd.read_csv(DATASET_REGIONAL)
df["unidad"] = (
    df["region"] + " | " + df["piso_ecologico"] + " (" + df["distrito"] + ")"
)
df["fecha"] = pd.to_datetime(
    df["anio"].astype(str) + "-" + df["numero_mes"].astype(str).str.zfill(2) + "-01"
)

CLIMA_EDA = [
    "temp_promedio", "precipitacion", "humedad_relativa",
    "radiacion_solar", "humedad_suelo",
]
MESES_ORDEN = [
    "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
    "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre",
]
df["mes"] = pd.Categorical(df["mes"], categories=MESES_ORDEN, ordered=True)

sns.set_theme(style="whitegrid", context="notebook")
print(f"Filas: {len(df):,} | Unidades: {df['unidad'].nunique()} | Regiones: {df['region'].nunique()}")
print(f"NaN produccion_piso_ton: {df['produccion_piso_ton'].isna().sum()}")
df.head(3)

## 1. Volumen productivo por unidad

Ranking de las 14 unidades (región × piso × distrito) según producción acumulada 2020–2025.

In [ ]:
vol_unidad = (
    df.groupby(["region", "piso_ecologico", "distrito", "unidad"], observed=True)["produccion_piso_ton"]
    .sum(min_count=1)
    .reset_index(name="produccion_total_ton")
    .sort_values("produccion_total_ton", ascending=False)
)
vol_unidad["pct_total"] = 100 * vol_unidad["produccion_total_ton"] / vol_unidad["produccion_total_ton"].sum()

print("=== Top 5 unidades por volumen ===")
print(vol_unidad.head(5)[["unidad", "produccion_total_ton", "pct_total"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=vol_unidad, y="unidad", x="produccion_total_ton",
    hue="region", dodge=False, legend=False, ax=ax,
)
ax.set_title("Producción acumulada 2020–2025 por unidad (Pareto-80)")
ax.set_xlabel("Toneladas")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_volumen_unidad.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Series temporales de producción

Agregación anual y mensual; líneas por unidad territorial.

In [ ]:
prod_anual = (
    df.groupby(["region", "piso_ecologico", "distrito", "unidad", "anio"], observed=True)["produccion_piso_ton"]
    .sum(min_count=1)
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=False)

for unidad, g in prod_anual.groupby("unidad"):
    axes[0].plot(g["anio"], g["produccion_piso_ton"], marker="o", label=unidad, linewidth=1.2)
axes[0].set_title("Producción anual agregada por unidad")
axes[0].set_xlabel("Año")
axes[0].set_ylabel("Toneladas")
axes[0].legend(bbox_to_anchor=(1.02, 1), fontsize=7, ncol=1)

for region, g in df.groupby("region"):
    m = g.groupby("fecha", observed=True)["produccion_piso_ton"].sum(min_count=1)
    axes[1].plot(m.index, m.values, label=region, linewidth=1.2)
axes[1].set_title("Producción mensual sumada por región (todas las unidades)")
axes[1].set_ylabel("Toneladas")
axes[1].legend(fontsize=8)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_produccion_anual.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Estacionalidad (mes × año)

Heatmaps por región: intensidad de cosecha mensual a lo largo del periodo.

In [ ]:
regiones = sorted(df["region"].unique())
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for ax, region in zip(axes, regiones):
    sub = df[df["region"] == region]
    pivot = sub.pivot_table(
        index="mes", columns="anio", values="produccion_piso_ton",
        aggfunc=lambda x: x.sum(min_count=1), observed=True,
    )
    pivot = pivot.reindex(MESES_ORDEN)
    sns.heatmap(pivot, ax=ax, cmap="YlOrRd", cbar_kws={"label": "ton"}, linewidths=0.2)
    ax.set_title(region)
    ax.set_xlabel("Año")
    ax.set_ylabel("Mes")

fig.suptitle("Estacionalidad productiva — mes × año (agregado regional)", y=1.02)
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_heatmap_estacionalidad.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Patrón estacional promedio

Promedio mensual de producción (todos los años) por unidad.

In [ ]:
patron = (
    df.groupby(["unidad", "mes"], observed=True)["produccion_piso_ton"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5))
for unidad, g in patron.groupby("unidad"):
    g = g.sort_values("mes")
    ax.plot(g["mes"].astype(str), g["produccion_piso_ton"], marker="o", label=unidad, linewidth=1)
ax.set_title("Patrón estacional promedio por unidad")
ax.set_xlabel("Mes")
ax.set_ylabel("Toneladas (promedio mensual)")
ax.tick_params(axis="x", rotation=45)
ax.legend(bbox_to_anchor=(1.02, 1), fontsize=7)
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_patron_estacional.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Perfil climático por piso

Medias 2020–2025 de variables climáticas core por unidad (descriptivo).

In [ ]:
clima_mean = (
    df.groupby(["unidad", "region"], observed=True)[CLIMA_EDA]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, len(CLIMA_EDA), figsize=(16, 5))
for ax, var in zip(axes, CLIMA_EDA):
    sns.barplot(
        data=clima_mean.sort_values(var, ascending=False),
        x=var, y="unidad", hue="region", dodge=False, ax=ax, legend=False,
    )
    ax.set_title(var)
    ax.set_ylabel("")
fig.suptitle("Perfil climático promedio por unidad (2020–2025)", y=1.02)
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_perfil_climatico.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df, x="region", y="temp_promedio", hue="piso_ecologico", ax=ax)
ax.set_title("Distribución mensual de temperatura por región y piso")
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_boxplot_temp.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Correlaciones exploratorias (Pearson)

Asociación mensual clima–producción **por unidad**. Sin corrección por comparaciones múltiples.

In [ ]:
rows = []
for unidad, g in df.groupby("unidad"):
    sub = g.dropna(subset=["produccion_piso_ton"] + CLIMA_EDA)
    if len(sub) < 12:
        continue
    region = g["region"].iloc[0]
    piso = g["piso_ecologico"].iloc[0]
    distrito = g["distrito"].iloc[0]
    for var in CLIMA_EDA:
        r, p = stats.pearsonr(sub["produccion_piso_ton"], sub[var])
        rows.append({
            "unidad": unidad, "region": region, "piso_ecologico": piso,
            "distrito": distrito, "variable_clima": var,
            "n": len(sub), "r": r, "p_valor": p,
        })

corr_regional = pd.DataFrame(rows)
corr_regional.to_csv(RUTA_OUTPUT / "eda_correlaciones_regional.csv", index=False, encoding="utf-8-sig")

top5 = corr_regional.reindex(corr_regional["r"].abs().sort_values(ascending=False).index).head(5)
print("=== Top 5 |r| por unidad (exploratorio, sin BH) ===")
print(top5[["unidad", "variable_clima", "n", "r", "p_valor"]].to_string(index=False))

sub_all = df.dropna(subset=["produccion_piso_ton"] + CLIMA_EDA)
print("\n=== Correlación agregada (pooling de todas las unidades) ===")
for var in CLIMA_EDA:
    r, p = stats.pearsonr(sub_all["produccion_piso_ton"], sub_all[var])
    print(f"{var:20s} r={r:+.3f}  p={p:.2e}  n={len(sub_all)}")

## 7. Caso sequía — Puno 2021→2022

Co-movimiento descriptivo entre producción agregada y humedad de suelo en altiplano (no atribución causal).

In [ ]:
puno = df[df["region"] == "Puno"].copy()

prod_puno = (
    puno.groupby(["unidad", "anio"], observed=True)["produccion_piso_ton"]
    .sum(min_count=1)
    .reset_index()
)
clima_puno = (
    puno.groupby(["unidad", "anio"], observed=True)["humedad_suelo"]
    .mean()
    .reset_index()
)

cambios = []
for unidad in prod_puno["unidad"].unique():
    p21 = prod_puno.query("unidad == @unidad and anio == 2021")["produccion_piso_ton"]
    p22 = prod_puno.query("unidad == @unidad and anio == 2022")["produccion_piso_ton"]
    if len(p21) and len(p22) and p21.iloc[0] > 0:
        pct = 100 * (p22.iloc[0] - p21.iloc[0]) / p21.iloc[0]
        cambios.append({"unidad": unidad, "pct_cambio_2021_2022": round(pct, 1)})
if cambios:
    print("=== Cambio producción Puno 2021→2022 ===")
    print(pd.DataFrame(cambios).to_string(index=False))

fig, ax1 = plt.subplots(figsize=(9, 4))
for unidad, g in prod_puno.groupby("unidad"):
    ax1.plot(g["anio"], g["produccion_piso_ton"], marker="o", label=f"Prod. {unidad}")
ax1.set_ylabel("Producción (ton)")
ax1.set_title("Puno — producción anual vs humedad de suelo (sequía altiplánica 2022)")
ax1.legend(fontsize=7, loc="upper left")

ax2 = ax1.twinx()
for unidad, g in clima_puno.groupby("unidad"):
    ax2.plot(g["anio"], g["humedad_suelo"], marker="s", linestyle="--", alpha=0.7, label=f"HS {unidad}")
ax2.set_ylabel("Humedad suelo (índice)")
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_puno_sequia.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Caso El Niño costero 2023–2024

Anomalías de precipitación y temperatura en Piura, La Libertad e Ica respecto al promedio 2020–2022.

In [ ]:
costa = df[df["region"].isin(["Piura", "La Libertad", "Ica"])].copy()
ref = costa[costa["anio"].between(2020, 2022)].groupby("region")[["precipitacion", "temp_promedio"]].mean()
evt = costa[costa["anio"].isin([2023, 2024])].groupby(["region", "anio"])[["precipitacion", "temp_promedio"]].mean()

print("=== Referencia 2020–2022 vs evento 2023–2024 (costa norte-centro) ===")
for region in ["Piura", "La Libertad", "Ica"]:
    if region not in ref.index:
        continue
    ref_p = ref.loc[region, "precipitacion"]
    ref_t = ref.loc[region, "temp_promedio"]
    print(f"\n{region} — ref precip={ref_p:.3f}  temp={ref_t:.2f}")
    if region in evt.index.get_level_values(0):
        print(evt.loc[region].round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for region, g in costa.groupby("region"):
    m = g.groupby("fecha", observed=True)["precipitacion"].mean()
    axes[0].plot(m.index, m.values, label=region, linewidth=1.2)
axes[0].axvspan(pd.Timestamp("2023-01-01"), pd.Timestamp("2024-12-31"), alpha=0.15, color="coral", label="2023–2024")
axes[0].set_title("Precipitación mensual — costa")
axes[0].set_ylabel("mm/día")
axes[0].legend(fontsize=8)

for region, g in costa.groupby("region"):
    m = g.groupby("fecha", observed=True)["produccion_piso_ton"].sum(min_count=1)
    axes[1].plot(m.index, m.values, label=region, linewidth=1.2)
axes[1].axvspan(pd.Timestamp("2023-01-01"), pd.Timestamp("2024-12-31"), alpha=0.15, color="coral")
axes[1].set_title("Producción mensual agregada — costa")
axes[1].set_ylabel("Toneladas")
axes[1].legend(fontsize=8)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(RUTA_FIGURES / "eda_regional_nino_costa.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Auditoría de calidad y cierre

- NaN en producción: meses sin cosecha reportada (política del pipeline, no imputados).
- **Siguiente paso:** `05_eda_por_cultivo.ipynb` desagrega por cultivo; `06_clustering_cultivos.ipynb` construye tipologías sobre perfiles Pareto-80.

In [ ]:
clima_cols = [c for c in df.columns if c.startswith("temp_") or c in (
    "precipitacion", "humedad_relativa", "radiacion_solar", "humedad_suelo",
)]
nan_audit = df[clima_cols + ["produccion_piso_ton"]].isna().sum()
print("=== Auditoría NaN ===")
print(nan_audit[nan_audit > 0] if (nan_audit > 0).any() else "Sin NaN en columnas revisadas")

resumen = {
    "filas": len(df),
    "unidades": df["unidad"].nunique(),
    "regiones": df["region"].nunique(),
    "nan_produccion": int(df["produccion_piso_ton"].isna().sum()),
    "figuras_generadas": len(list(RUTA_FIGURES.glob("eda_regional_*.png"))),
}
print("\n=== Resumen Análisis A ===")
for k, v in resumen.items():
    print(f"{k}: {v}")